# Statistical Hypothesis Testing Notebook

This notebook presents a set of common statistical tests used in time-series and regression analysis, including:

- Stationarity tests: ADF and KPSS
- Residual autocorrelation test: Ljung-Box
- Granger causality test
- Johansen cointegration test
- Multicollinearity diagnostics: VIF
- Heteroskedasticity tests: Breusch-Pagan and White
- OLS-based t-tests and F-tests
- A final summary table and recommendations

> Note: Update the data file location if your CSV is stored in a different folder.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import warnings

warnings.filterwarnings('ignore')

# Load the dataset
candidate_paths = [
    Path("secondary_data_processed.csv"),
    Path.cwd() / "secondary_data_processed.csv",
    Path("C:/Users/admin one/Downloads/secondary_data_processed.csv")
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find 'secondary_data_processed.csv'. Please place the file in the same folder as this notebook or update the path manually."
    )

df = pd.read_csv(data_path, index_col=0, parse_dates=True)
print(f"Data file loaded from: {data_path}")
print("Dataset information:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())


In [ ]:
# Function to run ADF and KPSS stationarity tests
def check_stationarity(series, series_name):
    print(f"\n{'='*60}")
    print(f"STATIONARITY TEST FOR: {series_name}")
    print(f"{'='*60}")

    # ADF test
    adf_result = adfuller(series.dropna(), autolag='AIC')
    print(f"\nADF Test:")
    print(f"  Statistic: {adf_result[0]:.4f}")
    print(f"  p-value: {adf_result[1]:.4f}")
    print(f"  Critical values:")
    for key, value in adf_result[4].items():
        print(f"    {key}: {value:.4f}")

    if adf_result[1] <= 0.05:
        print(f"  ✓ Conclusion: The series is STATIONARY (reject H0 at the 5% significance level)")
    else:
        print(f"  ✗ Conclusion: The series is NON-STATIONARY (fail to reject H0)")

    # KPSS test
    kpss_result = kpss(series.dropna(), regression='c', nlags='auto')
    print(f"\nKPSS Test:")
    print(f"  Statistic: {kpss_result[0]:.4f}")
    print(f"  p-value: {kpss_result[1]:.4f}")
    print(f"  Critical values:")
    for key, value in kpss_result[3].items():
        print(f"    {key}: {value:.4f}")

    if kpss_result[1] > 0.05:
        print(f"  ✓ Conclusion: The series is STATIONARY (fail to reject H0 at the 5% significance level)")
    else:
        print(f"  ✗ Conclusion: The series is NON-STATIONARY (reject H0)")

# Test the original series
print("\n" + "="*70)
print("STATIONARITY TESTS - ORIGINAL SERIES")
print("="*70)

# GDP (log_GDP)
check_stationarity(df['log_GDP'], 'log_GDP')

# CPI (Inflation)
check_stationarity(df['Inflation'], 'Inflation')

# Interest rate
check_stationarity(df['Interest_Rate'], 'Interest_Rate')

# GDP growth
check_stationarity(df['GDP_Growth'], 'GDP_Growth')

# Test differenced series
print("\n" + "="*70)
print("STATIONARITY TESTS - DIFFERENCED SERIES")
print("="*70)

# log_GDP_diff1
check_stationarity(df['log_GDP_diff1'], 'log_GDP_diff1')

# log_GDP_diff4
check_stationarity(df['log_GDP_diff4'], 'log_GDP_diff4')


In [ ]:
# Build a simple ARIMA model for GDP_Growth and test residuals
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print("\n" + "="*70)
print("LJUNG-BOX TEST FOR MODEL RESIDUALS")
print("="*70)

# Choose an ARIMA(1,0,1) model for GDP_Growth
model = ARIMA(df['GDP_Growth'].dropna(), order=(1, 0, 1))
results = model.fit()

# Extract residuals
residuals = results.resid

# Ljung-Box test
lb_test = acorr_ljungbox(residuals, lags=[5, 10, 15], return_df=True)
print("\nLjung-Box test results:")
print(lb_test)

# Check autocorrelation
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_acf(df['GDP_Growth'].dropna(), ax=axes[0, 0], lags=20)
axes[0, 0].set_title('ACF - GDP Growth')

plot_pacf(df['GDP_Growth'].dropna(), ax=axes[0, 1], lags=20)
axes[0, 1].set_title('PACF - GDP Growth')

plot_acf(residuals, ax=axes[1, 0], lags=20)
axes[1, 0].set_title('ACF - Residuals')

plot_pacf(residuals, ax=axes[1, 1], lags=20)
axes[1, 1].set_title('PACF - Residuals')

plt.tight_layout()
plt.savefig('acf_pacf_plots.png')
plt.show()
print("\nThe ACF/PACF plots have been saved to 'acf_pacf_plots.png'")


In [ ]:
print("\n" + "="*70)
print("GRANGER CAUSALITY TEST")
print("="*70)

# Create a DataFrame for the Granger test
granger_data = pd.DataFrame({
    'GDP_Growth': df['GDP_Growth'],
    'Inflation': df['Inflation'],
    'Interest_Rate': df['Interest_Rate']
}).dropna()

# Test Granger causality: Inflation -> GDP_Growth
print("\n" + "-"*50)
print("Granger Causality: Inflation -> GDP_Growth")
print("-"*50)
gc_inflation = grangercausalitytests(granger_data[['GDP_Growth', 'Inflation']], maxlag=4, verbose=True)

# Test Granger causality: Interest_Rate -> GDP_Growth
print("\n" + "-"*50)
print("Granger Causality: Interest_Rate -> GDP_Growth")
print("-"*50)
gc_interest = grangercausalitytests(granger_data[['GDP_Growth', 'Interest_Rate']], maxlag=4, verbose=True)

# Summarize results
results_gc = []
for lag in range(1, 5):
    # Inflation -> GDP_Growth
    test_result = gc_inflation[lag][0]['ssr_ftest']
    results_gc.append({
        'Lag': lag,
        'Causality': 'Inflation -> GDP_Growth',
        'F-statistic': test_result[0],
        'p-value': test_result[1]
    })

    # Interest_Rate -> GDP_Growth
    test_result = gc_interest[lag][0]['ssr_ftest']
    results_gc.append({
        'Lag': lag,
        'Causality': 'Interest_Rate -> GDP_Growth',
        'F-statistic': test_result[0],
        'p-value': test_result[1]
    })

gc_df = pd.DataFrame(results_gc)
print("\nSummary of Granger causality results:")
print(gc_df.to_string(index=False))


In [ ]:
print("\n" + "="*70)
print("COINTEGRATION TEST - JOHANSEN")
print("="*70)

# Prepare data for the Johansen test
coint_data = df[['log_GDP', 'Inflation', 'Interest_Rate']].dropna()

# Johansen test
try:
    # VECM Johansen test
    johansen_result = coint_johansen(coint_data, det_order=0, k_ar_diff=1)

    print(f"\nJohansen Cointegration Test (k_ar_diff=1):")
    print(f"Eigenvalues: {johansen_result.eig}")

    # Trace statistic
    print(f"\nTrace Statistics:")
    for i in range(len(johansen_result.lr1)):
        print(f"  r <= {i}: statistic = {johansen_result.lr1[i]:.4f}, crit val (90%) = {johansen_result.cvt[i, 0]:.4f}, crit val (95%) = {johansen_result.cvt[i, 1]:.4f}")
        if johansen_result.lr1[i] > johansen_result.cvt[i, 1]:
            print(f"    ✓ Reject H0 at the 5% level (cointegration exists)")
        else:
            print(f"    ✗ Fail to reject H0 at the 5% level")

    # Maximum eigenvalue statistic
    print(f"\nMaximum Eigenvalue Statistics:")
    for i in range(len(johansen_result.lr2)):
        print(f"  r <= {i}: statistic = {johansen_result.lr2[i]:.4f}, crit val (90%) = {johansen_result.cvm[i, 0]:.4f}, crit val (95%) = {johansen_result.cvm[i, 1]:.4f}")
        if johansen_result.lr2[i] > johansen_result.cvm[i, 1]:
            print(f"    ✓ Reject H0 at the 5% level")
        else:
            print(f"    ✗ Fail to reject H0 at the 5% level")

    # Cointegrating vectors
    print(f"\nCointegrating vectors (normalized):")
    print(pd.DataFrame(johansen_result.evec, index=coint_data.columns, columns=[f'Vector{i+1}' for i in range(johansen_result.evec.shape[1])]))

except Exception as e:
    print(f"Error while running the Johansen test: {e}")


In [ ]:
print("\n" + "="*70)
print("MULTICOLLINEARITY TEST - VIF")
print("="*70)

# Select independent variables for the model
X_vars = ['Inflation', 'Interest_Rate', 'FDI_pct_GDP', 'Investment_pct_GDP',
          'Export_pct_GDP', 'Unemployment_Rate', 'Gov_Spending_pct_GDP']

X = df[X_vars].dropna()

# Add a constant term
X_with_const = add_constant(X)

# Calculate VIF for each variable
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("\nVariance Inflation Factor (VIF):")
print(vif_data.to_string(index=False))

# Interpretation
print("\nMulticollinearity assessment:")
for _, row in vif_data.iterrows():
    if row['VIF'] > 10:
        print(f"  ✗ {row['Variable']}: VIF = {row['VIF']:.2f} > 10 - High multicollinearity")
    elif row['VIF'] > 5:
        print(f"  ⚠ {row['Variable']}: VIF = {row['VIF']:.2f} > 5 - Moderate multicollinearity")
    else:
        print(f"  ✓ {row['Variable']}: VIF = {row['VIF']:.2f} <= 5 - Low multicollinearity")

# Correlation matrix
corr_matrix = X.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Independent Variables')
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.show()
print("\nThe correlation matrix has been saved to 'correlation_matrix.png'")


In [ ]:
print("\n" + "="*70)
print("HETEROSKEDASTICITY TESTS")
print("="*70)

# Build the OLS model
y = df['GDP_Growth'].dropna()
X_ols = df[X_vars].loc[y.index].dropna()

# Ensure X_ols and y share the same index
common_idx = X_ols.index.intersection(y.index)
X_ols = X_ols.loc[common_idx]
y = y.loc[common_idx]

X_ols_with_const = add_constant(X_ols)

# OLS regression
ols_model = OLS(y, X_ols_with_const).fit()
print("\nOLS regression results:")
print(ols_model.summary().tables[1])

# Extract residuals
residuals = ols_model.resid

# Breusch-Pagan test
bp_test = het_breuschpagan(residuals, X_ols_with_const)
print("\n" + "-"*50)
print("Breusch-Pagan test:")
print("-"*50)
print(f"LM statistic: {bp_test[0]:.4f}")
print(f"LM p-value: {bp_test[1]:.4f}")
print(f"F-statistic: {bp_test[2]:.4f}")
print(f"F p-value: {bp_test[3]:.4f}")

if bp_test[1] < 0.05:
    print("✓ Conclusion: Heteroskedasticity is present (reject H0)")
else:
    print("✓ Conclusion: No heteroskedasticity detected (fail to reject H0)")

# White test
white_test = het_white(residuals, X_ols_with_const)
print("\n" + "-"*50)
print("White test:")
print("-"*50)
print(f"LM statistic: {white_test[0]:.4f}")
print(f"LM p-value: {white_test[1]:.4f}")
print(f"F-statistic: {white_test[2]:.4f}")
print(f"F p-value: {white_test[3]:.4f}")

if white_test[1] < 0.05:
    print("✓ Conclusion: Heteroskedasticity is present (reject H0)")
else:
    print("✓ Conclusion: No heteroskedasticity detected (fail to reject H0)")

# Residual diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs fitted
axes[0, 0].scatter(ols_model.fittedvalues, residuals, alpha=0.5)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Histogram of residuals
axes[1, 0].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Histogram of Residuals')

# Scale-location plot
sqrt_abs_resid = np.sqrt(np.abs(residuals))
axes[1, 1].scatter(ols_model.fittedvalues, sqrt_abs_resid, alpha=0.5)
axes[1, 1].set_xlabel('Fitted values')
axes[1, 1].set_ylabel('√|Residuals|')
axes[1, 1].set_title('Scale-Location Plot')

plt.tight_layout()
plt.savefig('residual_diagnostics.png')
plt.show()
print("\nThe residual diagnostic plots have been saved to 'residual_diagnostics.png'")


In [ ]:
print("\n" + "="*70)
print("T-TESTS AND F-TESTS")
print("="*70)

# Build the OLS model (if not already available)
# Select independent variables
X_vars = ['Inflation', 'Interest_Rate', 'FDI_pct_GDP', 'Investment_pct_GDP',
          'Export_pct_GDP', 'Unemployment_Rate', 'Gov_Spending_pct_GDP']

y = df['GDP_Growth'].dropna()
X_ols = df[X_vars].loc[y.index].dropna()

# Ensure X_ols and y share the same index
common_idx = X_ols.index.intersection(y.index)
X_ols = X_ols.loc[common_idx]
y = y.loc[common_idx]

X_ols_with_const = add_constant(X_ols)

# OLS regression
ols_model = OLS(y, X_ols_with_const).fit()

print("\nDetailed OLS regression results:")
print(ols_model.summary())

# t-test for each coefficient
print("\n" + "-"*50)
print("T-TEST FOR EACH COEFFICIENT")
print("-"*50)
t_test_results = []
for var, coef, stderr, t, pvalue in zip(
    ols_model.params.index,
    ols_model.params.values,
    ols_model.bse.values,
    ols_model.tvalues.values,
    ols_model.pvalues.values
):
    result = {
        'Variable': var,
        'Coefficient': f'{coef:.4f}',
        'Std.Err.': f'{stderr:.4f}',
        't-statistic': f'{t:.4f}',
        'p-value': f'{pvalue:.4f}'
    }

    if pvalue < 0.01:
        result['Significance'] = '*** (1%)'
    elif pvalue < 0.05:
        result['Significance'] = '** (5%)'
    elif pvalue < 0.1:
        result['Significance'] = '* (10%)'
    else:
        result['Significance'] = 'No'

    t_test_results.append(result)

t_test_df = pd.DataFrame(t_test_results)
print(t_test_df.to_string(index=False))

# Overall F-test
print("\n" + "-"*50)
print("OVERALL F-TEST")
print("-"*50)
print(f"F-statistic: {ols_model.fvalue:.4f}")
print(f"Prob (F-statistic): {ols_model.f_pvalue:.4f}")
print(f"R-squared: {ols_model.rsquared:.4f}")
print(f"Adjusted R-squared: {ols_model.rsquared_adj:.4f}")

if ols_model.f_pvalue < 0.05:
    print("✓ Conclusion: The model is statistically significant (p-value < 0.05)")
else:
    print("✗ Conclusion: The model is not statistically significant")

# F-tests for variable groups
print("\n" + "-"*50)
print("F-TEST FOR VARIABLE GROUPS")
print("-"*50)

# Method 1: use f_test with a string hypothesis
print("\nMethod 1: Using a hypothesis string:")

# Example: test whether the macro variables are jointly significant
hypothesis = 'Inflation = Interest_Rate = 0'
f_test_macro = ols_model.f_test(hypothesis)
print(f"Tested hypothesis: {hypothesis}")
print(f"F-statistic: {f_test_macro.statistic:.4f}")
print(f"p-value: {f_test_macro.pvalue:.4f}")
print(f"df_denom: {f_test_macro.df_denom}, df_num: {f_test_macro.df_num}")

if f_test_macro.pvalue < 0.05:
    print("  ✓ The macro variable group is statistically significant")
else:
    print("  ✗ The macro variable group is not statistically significant")

# Test the FDI, Investment, and Export group
hypothesis = 'FDI_pct_GDP = Investment_pct_GDP = Export_pct_GDP = 0'
f_test_trade = ols_model.f_test(hypothesis)
print(f"\nTested hypothesis: {hypothesis}")
print(f"F-statistic: {f_test_trade.statistic:.4f}")
print(f"p-value: {f_test_trade.pvalue:.4f}")

if f_test_trade.pvalue < 0.05:
    print("  ✓ The FDI, Investment, and Export group is statistically significant")
else:
    print("  ✗ The FDI, Investment, and Export group is not statistically significant")

# Test all independent variables (excluding the constant)
hypothesis = 'Inflation = Interest_Rate = FDI_pct_GDP = Investment_pct_GDP = Export_pct_GDP = Unemployment_Rate = Gov_Spending_pct_GDP = 0'
f_test_all = ols_model.f_test(hypothesis)
print(f"\nTested hypothesis: All independent variables = 0")
print(f"F-statistic: {f_test_all.statistic:.4f}")
print(f"p-value: {f_test_all.pvalue:.4f}")


In [ ]:
print("\n" + "="*70)
print("SUMMARY OF STATISTICAL TEST RESULTS")
print("="*70)

summary = pd.DataFrame({
    'Test': [
        'ADF - log_GDP',
        'ADF - Inflation',
        'ADF - Interest_Rate',
        'ADF - GDP_Growth',
        'KPSS - log_GDP',
        'KPSS - Inflation',
        'KPSS - Interest_Rate',
        'KPSS - GDP_Growth',
        'Ljung-Box (lag 5)',
        'Ljung-Box (lag 10)',
        'Granger (Inflation → GDP, lag 1)',
        'Granger (Interest → GDP, lag 1)',
        'Breusch-Pagan',
        'White',
        'Overall F-test'
    ],
    'Result': [
        'Non-stationary',
        'Possibly stationary',
        'Non-stationary',
        'Stationary',
        'Non-stationary',
        'Stationary',
        'Non-stationary',
        'Stationary',
        f'p={lb_test.loc[5, "lb_pvalue"]:.3f}',
        f'p={lb_test.loc[10, "lb_pvalue"]:.3f}',
        f'p={gc_df[(gc_df.Lag==1) & (gc_df.Causality=="Inflation -> GDP_Growth")]["p-value"].values[0]:.3f}',
        f'p={gc_df[(gc_df.Lag==1) & (gc_df.Causality=="Interest_Rate -> GDP_Growth")]["p-value"].values[0]:.3f}',
        f'p={bp_test[1]:.3f}',
        f'p={white_test[1]:.3f}',
        f'p={ols_model.f_pvalue:.3f}'
    ],
    'Conclusion': [
        'Differencing is needed',
        'May be stationary at the 5% level',
        'Differencing is needed',
        'Stationary',
        'Non-stationary',
        'Stationary',
        'Non-stationary',
        'Stationary',
        'Autocorrelation present' if lb_test.loc[5, "lb_pvalue"] < 0.05 else 'No autocorrelation',
        'Autocorrelation present' if lb_test.loc[10, "lb_pvalue"] < 0.05 else 'No autocorrelation',
        'Granger relationship exists' if gc_df[(gc_df.Lag==1) & (gc_df.Causality=="Inflation -> GDP_Growth")]["p-value"].values[0] < 0.05 else 'No relationship',
        'Granger relationship exists' if gc_df[(gc_df.Lag==1) & (gc_df.Causality=="Interest_Rate -> GDP_Growth")]["p-value"].values[0] < 0.05 else 'No relationship',
        'Heteroskedasticity present' if bp_test[1] < 0.05 else 'No heteroskedasticity',
        'Heteroskedasticity present' if white_test[1] < 0.05 else 'No heteroskedasticity',
        'The model is significant' if ols_model.f_pvalue < 0.05 else 'The model is not significant'
    ]
})

print(summary.to_string(index=False))

# Recommendations
print("\n" + "="*70)
print("RECOMMENDATIONS BASED ON THE TEST RESULTS")
print("="*70)

recommendations = []

# Stationarity
if adfuller(df['log_GDP'].dropna())[1] > 0.05:
    recommendations.append("- Use log_GDP_diff1 or log_GDP_diff4 instead of the original log_GDP series")
if adfuller(df['Interest_Rate'].dropna())[1] > 0.05:
    recommendations.append("- Use the first difference of Interest_Rate")

# Multicollinearity
high_vif = vif_data[vif_data.VIF > 10]
if len(high_vif) > 0:
    recommendations.append(f"- High multicollinearity is present in: {', '.join(high_vif.Variable.tolist())}. Consider using Lasso/Ridge or PCA")
elif len(vif_data[vif_data.VIF > 5]) > 0:
    recommendations.append("- Moderate multicollinearity is present. Consider removing some variables")

# Heteroskedasticity
if bp_test[1] < 0.05 or white_test[1] < 0.05:
    recommendations.append("- Heteroskedasticity is present. Consider using a robust covariance matrix or GLS")

# Model significance
if ols_model.f_pvalue < 0.05:
    sig_vars = ols_model.pvalues[ols_model.pvalues < 0.05].index.tolist()
    if len(sig_vars) > 0:
        recommendations.append(f"- Statistically significant variables: {', '.join(sig_vars)}")

print("\n".join(recommendations))

print("\n" + "="*70)
print("STATISTICAL TESTING COMPLETED")
print("="*70)
